# EcoWatt Energy Analysis System

This notebook analyzes household electricity consumption using a rule-based model.  
It compares user consumption against similar Saudi households and estimates potential savings.

The system:
- Loads baseline data  
- Processes household bills  
- Estimates baseline consumption  
- Calculates optimized consumption  
- Identifies consumption drivers  
- Selects relevant recommendations from a predefined tips dataset (rule-based selection)  
- Visualizes trends and comparisons  

In [ ]:
#------ Imports ------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar

plt.style.use('default')

## SaudiElectricityAnalyzer Class

This class:

- Loads and preprocesses baseline and tips datasets  
- Cleans and normalizes input data  
- Compares household consumption with similar homes  
- Estimates baseline and optimized electricity usage  
- Calculates potential savings  
- Identifies key consumption drivers  
- Selects recommendations from a predefined dataset using rule-based filtering  

In [ ]:
#------ Class Definition ------

class SaudiElectricityAnalyzer:

    def __init__(self, baseline_path, tips_path):

        self.baseline_df = pd.read_csv(baseline_path)
        self.tips_df = pd.read_excel(tips_path)

        self.baseline_df['Region'] = self.baseline_df['Region'].str.lower()
        self.baseline_df['Season'] = self.baseline_df['Season'].str.lower()
        self.baseline_df['Dwelling_Type'] = self.baseline_df['Dwelling_Type'].str.lower()

    @staticmethod
    def estimate_cost(kwh):
        if kwh <= 6000:
            base_cost = kwh * 0.18
        else:
            base_cost = (6000 * 0.18) + ((kwh - 6000) * 0.30)

        # Add 15% Value Added Tax (VAT) and the 15 SAR fixed meter reading fee
        total_with_vat = (base_cost * 1.15) + 15

        return total_with_vat


    def analyze_household(self, household_bills):

        household_bills = household_bills.copy()

        household_bills['region'] = household_bills['region'].str.lower()
        household_bills['season'] = household_bills['season'].str.lower()
        household_bills['housing_type'] = household_bills['housing_type'].str.lower()

        # normalize insulation early
        household_bills['has_insulation'] = household_bills['has_insulation'].replace({
            'TRUE': True, 'FALSE': False,
            'True': True, 'False': False
        }).astype(bool)

        merged = pd.merge(
            household_bills,
            self.baseline_df,
            left_on=['region', 'season', 'housing_type'],
            right_on=['Region', 'Season', 'Dwelling_Type'],
            how='left'
        )

        if 'ac_type' not in merged.columns:
            merged['ac_type'] = 'central'

        # Estimated average operating power derived from typical SEER ranges of SASO-compliant units
        power_split_ac = 2.2
        power_window_ac = 3.2
        power_central_ac = 2.6

        power_heater = 1.5
        power_water_heater = 2.0

        power_lamp_reg = 0.06
        power_lamp_sav = 0.009
        power_fridge = 0.15
        power_freezer = 0.12


        # --- AC efficiency by type ---
        merged['ac_type'] = merged['ac_type'].astype(str).str.lower()

        # separate AC and heater usage explicitly

        # Estimate number of AC units (1 AC per ~2 people, rounded)
        estimated_ac_units = np.round(merged['number_of_residents'] / 2)

        # Prevent zero AC units
        estimated_ac_units = np.maximum(estimated_ac_units, 1)

        # Select correct AC hours and matching power rating based on ac_type
        # Default is Split AC
        ac_hours = np.select(
            [
                merged['ac_type'] == 'window',
                merged['ac_type'] == 'central'
            ],
            [
                merged['Window_AC_hours_per_week'],
                merged['Central_AC_hours_per_week']
            ],
            default=merged['Split_AC_hours_per_week']
        )

        ac_power = np.select(
            [
                merged['ac_type'] == 'window',
                merged['ac_type'] == 'central'
            ],
            [
                power_window_ac,
                power_central_ac
            ],
            default=power_split_ac
        )


        # Adjust AC consumption using the specific power rating for the AC type
        ac_usage = ac_hours * ac_power * estimated_ac_units

        heater_usage = merged['Radiant_Heater_hours_per_week'] * power_heater

        # Water heating + lighting + refrigeration base load
        base = (merged['Water_Heater_hours_per_week'] * power_water_heater) + \
               (merged['Regular_Lamps_hours_per_week'] * power_lamp_reg) + \
               (merged['Energy_Saving_Lamps_hours_per_week'] * power_lamp_sav) + \
               (merged['Refrigerator_hours_per_week'] * power_fridge) + \
               (merged['Freezer_hours_per_week'] * power_freezer)

        # Ensures temperature completeness
        merged['avg_temp'] = merged['avg_temp'].fillna(merged['Average_Temp_C'])
        merged['Average_Temp_C'] = merged['Average_Temp_C'].fillna(merged['avg_temp'])


        # Saudi regional cooling sensitivity (KAPSARC study)
        # Values represent % increase in cooling demand per °C
        region_sensitivity = {

            # Central
            'riyadh': 0.083,
            'qassim': 0.083,
            'hail': 0.083,

            # Western
            'makkah': 0.129,
            'madinah': 0.129,
            'tabuk': 0.129,

            # Southern
            'aseer': 0.103,
            'jazan': 0.103,
            'najran': 0.103,
            'al-baha': 0.103,

            # Eastern
            'eastern region': 0.048,

            # Northern
            'northern borders': 0.083,
            'al-jouf': 0.083
        }

        merged['cooling_sensitivity'] = merged['region'].map(region_sensitivity)

        insulation_rate = merged['Insulation_Yes_Pct'] / 100
        insulation_rate = insulation_rate.fillna(0.5)

        merged['insulation_rate'] = insulation_rate

        # --- Region-based cooling adjustment only ---
        region_factor = 1 + merged['cooling_sensitivity'].fillna(0)

        # Apply insulation once only
        insulation_factor = 1 - (merged['insulation_rate'] * 0.15)

        ac_adjusted = ac_usage * region_factor * insulation_factor
        heater_adjusted = heater_usage * insulation_factor

        hvac_adjusted = ac_adjusted + heater_adjusted

        # Calculate final weekly kWh
        weekly_kwh = hvac_adjusted + base

        # Occupancy scaling (~1–3% per additional person)
        occupancy_factor = 1 + (merged['number_of_residents'] - 4) * 0.015
        occupancy_factor = np.clip(occupancy_factor, 0.95, 1.15)

        weekly_kwh = weekly_kwh * occupancy_factor

        monthly_partial = weekly_kwh * 4.345  # weeks/month (calendar average)
        merged['baseline_kwh'] = monthly_partial

        # Prevent unrealistically low baseline
        merged['baseline_kwh'] = np.maximum(merged['baseline_kwh'], 200)

        #------ Remaining Analysis Logic ------

        # Behavioral baseline saving (~3%) even for efficient homes
        # Source: IEA, Energy.gov → behavioral savings typically 3–10%
        r1 = 0.03

        # Insulation impact:
        # Studies (SEEC, KSU) show 30–40% reduction in cooling demand
        # Effective total reduction ≈ ~15%
        r2 = np.where(merged['has_insulation'] == False, 0.15, 0)

        # AC efficiency gap: ~5–15% → using 8% (IEA, SASO)
        r3 = np.select(
            [
                merged['ac_type'] == 'window',
                merged['ac_type'] == 'central'
            ],
            [
                0.08,  # window - DOE / Energy Star references
                0.03   # central - conservative improvement range
            ],
            default=0
        )

        # Occupancy effect (~2–5%)
        r4 = (merged['number_of_residents'] - 4) * 0.02
        r4 = np.clip(r4, 0, 0.05)

        merged['reduction'] = 1 - (1 - r1) * (1 - r2) * (1 - r3) * (1 - r4)

        MAX_REDUCTION = 0.25  # Heuristic cap to constrain compounded reduction effects and ensure realistic savings
        merged['reduction'] = merged['reduction'].clip(upper=MAX_REDUCTION)

        # Apply the reduction percentage based on actual issues (insulation, AC, occupancy)
        merged['optimized_kwh'] = merged['total_kwh'] * (1 - merged['reduction'])

        # Calculate theoretical costs for a fair comparison
        theoretical_actual_cost = merged['total_kwh'].apply(self.estimate_cost)
        merged['optimized_cost'] = merged['optimized_kwh'].apply(self.estimate_cost)

        # Keep the actual cost for user display
        merged['actual_cost'] = np.where(
            merged['total_cost_sar'].notna(),
            merged['total_cost_sar'],
            theoretical_actual_cost
        )

        # 3. Final Saving Calculation
        # Savings = difference between current theoretical cost and optimized cost
        merged['saving_sar'] = (theoretical_actual_cost - merged['optimized_cost']).clip(lower=0)

        merged = merged.sort_values(['year', 'month'])

        def build_row(row):

            drivers = []

            if pd.notna(row['baseline_kwh']):

                if not row['has_insulation']:
                    drivers.append("Poor insulation")

                if str(row['ac_type']).lower() == 'window':
                    drivers.append("Low efficiency AC")

                if row['number_of_residents'] >= 6:
                    drivers.append("High occupancy")

            if pd.isna(row['baseline_kwh']):
                msg = "No baseline available"
                level = "Average"
            else:
                baseline_safe = max(row['baseline_kwh'], 1)
                diff = ((row['total_kwh'] - baseline_safe) / baseline_safe) * 100

                if diff > 20:
                    msg = f"Significantly above similar homes (+{diff:.0f}%)"
                    level = "High"
                elif diff > 5:
                    msg = f"Slightly above similar homes (+{diff:.0f}%)"
                    level = "Moderate"
                elif diff < -10:
                    msg = f"Efficient compared to similar homes (-{abs(diff):.0f}%)"
                    level = "Low"
                else:
                    msg = "Within the average range of similar homes"
                    level = "Average"

            return pd.Series({
                'drivers': drivers,
                'message': msg,
                'consumption_level': level
            })

        merged[['drivers', 'message', 'consumption_level']] = merged.apply(build_row, axis=1)

        results = {
            'household_id': household_bills['household_id'].iloc[0],
            'trend_data': []
        }

        for _, row in merged.iterrows():

            month_label = f"{calendar.month_abbr[int(row['month'])]} '{str(row['year'])[-2:]}"

            results['trend_data'].append({
                'month_label': month_label,
                'actual_kwh': row['total_kwh'],
                'optimized_kwh': row['optimized_kwh'],
                'similar_homes_kwh': row['baseline_kwh'],
                'drivers': row['drivers'],
                'message': row['message'],
                'consumption_level': row['consumption_level'],
                'saving_sar': row['saving_sar']
            })

        results['advice'] = self._generate_advice(results)

        return results

    #------ Advice Function ------

    def _generate_advice(self, results):

        latest = results['trend_data'][-1]
        advice_list = []

        level = latest['consumption_level']
        drivers = latest.get('drivers', [])

        mapping = {
            "Poor insulation": "insulation",
            "Low efficiency AC": "cooling",
            "High occupancy": "behavior",
            "High cooling demand": "cooling"
        }

        for d in drivers:
            if d in mapping:
                tips = self.tips_df[self.tips_df['type'] == mapping[d]]
                if not tips.empty:
                    n = 3 if level == "High" else 2
                    advice_list.extend(tips.sample(n=min(n, len(tips))).to_dict('records'))

        if level == "High":
            cats = ['cooling', 'electronics', 'lighting']
        elif level == "Moderate":
            cats = ['cooling', 'lighting']
        else:
            cats = ['behavior', 'lighting']

        for cat in cats:
            tips = self.tips_df[self.tips_df['type'] == cat]
            if not tips.empty:
                n = 3 if level == "High" else 2
                advice_list.extend(tips.sample(n=min(n, len(tips))).to_dict('records'))

        seen = set()
        final = []

        for tip in advice_list:
            if tip['advice'] not in seen:
                seen.add(tip['advice'])
                final.append({
                    'title': tip['type'],
                    'advice': tip['advice'],
                    'source': tip['reference']
                })

        return final[:6]

#------ Plot Functions ------

def plot_trend_chart(trend_data):

    months = [x['month_label'] for x in trend_data]
    actuals = [x['actual_kwh'] for x in trend_data]
    optimized = [x['optimized_kwh'] for x in trend_data]

    PRIMARY_GREEN = '#10B981'
    MUTED_GREEN = '#6EE7B7'

    plt.figure(figsize=(9, 5))
    if len(trend_data) == 1:
        plt.plot(months, actuals, color=PRIMARY_GREEN, marker='o', label='Actual')
        plt.plot(months, optimized, color=MUTED_GREEN, linestyle='--', marker='o', label='Potential saving')
    else:
        plt.plot(months, actuals, color=PRIMARY_GREEN, marker='o', label='Actual')
        plt.plot(months, optimized, color=MUTED_GREEN, linestyle='--', label='Potential saving')

    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    plt.close()


def plot_comparison_chart(latest):

    PRIMARY_GREEN = '#10B981'

    plt.figure()
    plt.bar(
        ['Your Home', 'Similar Saudi Homes'],
        [latest['actual_kwh'], latest['similar_homes_kwh']],
        color=PRIMARY_GREEN
    )
    plt.show()
    plt.close()


#------ Load Data ------
analyzer = SaudiElectricityAnalyzer(
    'household_consumption_baseline.csv',
    'EcowattTips.xlsx'
)

user_df = pd.read_csv('user_profile.csv')


#------ Execution ------

print("*" * 44)
print("EcoWatt Rule-Based Energy Analysis System")
print("Using GASTAT Saudi Household Baseline Data")
print("*" * 44)

for hid, bills in user_df.groupby('household_id'):

    print("\n" + "=" * 20)
    print(f"Household #{hid}")
    print("=" * 20)

    results = analyzer.analyze_household(bills)

    print("\n" + "-" * 52)
    print("6-Month Consumption Trend (Actual vs Potential Saving)")
    print("-" * 52)

    trend_data = results['trend_data']
    last_6_months = trend_data[-6:] if len(trend_data) >= 6 else trend_data
    plot_trend_chart(last_6_months)

    latest = results['trend_data'][-1]

    print("\n" + "-" * 37)
    print("Comparison with Similar Saudi Homes")
    print("-" * 37)

    print("\nAnalysis:", latest['message'])
    print("Drivers:", latest['drivers'])

    plot_comparison_chart(latest)

    print("\n" + "-" * 15)
    print("EcoWatt Advice")
    print("-" * 15)

    print("\nAdvice:")
    for tip in results['advice']:
        print("-", tip['advice'])

    print(f"\nEstimated Saving: {latest['saving_sar']:.2f} SAR")